# Exercise 3 — Feature selection

Last week we measured k-NN on unseen data. Today we ask **which columns it
needs**: first by hand, then in NumPy, then with scikit-learn.

A short pandas warm-up comes first; keep it open as a reference.

1. Prepare the data in pandas (Task 1).
2. Understand filter scores by hand, then implement them (Tasks 2–3).
3. Select features with a validation score and your own loop (Tasks 4, 6–7).

Task 5 (PCA) is an optional implementation task; skip it in the main session.

Keep AI assistance switched off, as in the first two labs.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/tomasvicar/MLR-public/master/exercises/data/"

## Pandas warm-up — a small table

Run these examples before Task 1, then use them as a reference. A **DataFrame**
is a table; one column is a **Series**. Here we create a tiny table of parcels.
To load a CSV instead, use `pd.read_csv("file.csv")` or `pd.read_csv(url)`,
as in Task 1. `np.nan` means a missing value.

In [ ]:
demo = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "weight_kg": [1.0, 2.0, np.nan, 4.0, 500.0, 3.0],
    "price": [10.0, np.nan, 30.0, 40.0, 20.0, 30.0],
    "rating": ["low", "high", "medium", "low", "high", "medium"],
    "depot": ["A", "B", "A", "B", "A", "B"],
    "delivered": ["no", "yes", "no", "yes", "no", np.nan],
})
display(demo.head())             # first five rows
print(demo.isna().sum())          # count missing values in each column

**Select and inspect.** One column name returns a Series; a list of names
returns a DataFrame. A histogram shows the distribution of a numeric column.

In [ ]:
display(demo["weight_kg"])
display(demo[["weight_kg", "price"]])
print("rows:", len(demo), "columns:", list(demo.columns))
demo["weight_kg"].hist(bins=5)
plt.xlabel("weight [kg]")
plt.ylabel("parcels")
plt.show()

**Copy and filter rows.** Keep the original table with `.copy()`.
`dropna(subset=[...])` removes rows missing a value in the named columns.
A comparison makes a boolean mask: `~` means NOT, `&` means AND.
Put each comparison in parentheses. Here the fixed limits keep missing
measurements for the next step.

In [ ]:
clean = demo.copy()
clean = clean.dropna(subset=["delivered"])
keep = ~(clean["weight_kg"] > 50) & ~(clean["price"] > 100)
clean = clean[keep]
display(clean)

**Encode categories.** `.map()` replaces each value using a dictionary;
assigning back with `clean["column"] = ...` updates the column. Ratings have
an order; depots do not, so `get_dummies` makes a separate 0/1 column per depot.

In [ ]:
clean["delivered"] = clean["delivered"].map({"no": 0, "yes": 1})
clean["rating"] = clean["rating"].map({"low": 1, "medium": 2, "high": 3})
clean = pd.get_dummies(clean, columns=["depot"], dtype=int)
display(clean)

**Fill missing values.** `.mean()` and `.median()` ignore missing entries.
`fillna` accepts one value or a dictionary giving a value for each column.
The following treats `clean` as a tiny **training table**. In Task 1, compute
the replacements from `train` only and reuse them for validation and test.

In [ ]:
print(clean["weight_kg"].mean(), clean["price"].median())
# Syntax example: replace missing entries in one column with a constant.
print(clean["price"].fillna(0))

# For our table, choose the training mean and median instead.
replacements = {"weight_kg": clean["weight_kg"].mean(),
                "price": clean["price"].median()}
clean = clean.fillna(replacements)
display(clean)

**From pandas to NumPy.** `.drop(columns=[...])` removes columns;
`.to_numpy()` returns their values as an array. Leave out identifiers and
keep the target separate from the input features.

In [ ]:
demo_features = clean.drop(columns=["id", "delivered"])
demo_names = list(demo_features.columns)
demo_X = demo_features.to_numpy(dtype=float)
demo_y = clean["delivered"].to_numpy()
print(demo_names)
print(demo_X)
print(demo_y)

## Task 1 — The table in pandas

We want to predict `passed_exam` from the other columns of a table of 100
students. Inspect the missing values and the height histogram. Which values
cannot be real measurements?

In [ ]:
raw = pd.read_csv(DATA + "ex03_students.csv")
display(raw.head())
print(raw.isna().sum())
raw["height_cm"].hist(bins=15)
plt.xlabel("height [cm]")
plt.ylabel("students")
plt.show()

Remove rows with unknown `gender` or impossible measurements
(`height_cm > 250`, `weight_kg > 200`). Keep missing measurements for now.
Encode `gender` as M = 0, F = 1, and `study_effort` as
low = 1, medium = 2, high = 3, very-high = 4.

The city names have no order, so their one-hot encoding is provided.
`student_id` is an identifier, not a measurement; leave it out.

In [ ]:
df = raw.copy()
# TODO: your code here
...

df = pd.get_dummies(df, columns=["city"], dtype=int)
features = df.drop(columns=["student_id", "passed_exam"])
feature_names = list(features.columns)
y = df["passed_exam"].to_numpy()
print("rows:", len(df), "features:", feature_names)

The split is provided, as in lab 2. **Use only the training rows** to
compute the replacement mean for age and weight and the median for height.
Use 0 for missing test scores: for this exercise we assume they mean a test
was not taken. That assumption would need checking with the data owner.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

train, test, y_train, y_test = train_test_split(
    features, y, test_size=0.2, random_state=42, stratify=y)
train, valid, y_train, y_valid = train_test_split(
    train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# TODO: your code here
fill_values = {}
train = train.fillna(fill_values)
valid = valid.fillna(fill_values)
test = test.fillna(fill_values)

scaler = StandardScaler().fit(train)
X_train = scaler.transform(train)
X_valid = scaler.transform(valid)
X_test = scaler.transform(test)
print("train / validation / test:", len(train), len(valid), len(test))

**Checkpoint before continuing:** 83 cleaned rows, 10 features, and
52 / 14 / 17 training / validation / test rows. After filling missing values,
`train.isna().sum().sum()` should be 0 (also for `valid` and `test`).

## Task 2 — Filter scores by hand

**Together at the board, about 15 minutes.** Four invented students;
`effort` means high study effort, `brno` means living in Brno, and `passed`
is the label. This deliberately simple table illustrates the formulas.

| # | `effort` | `brno` | `passed` |
|---|---:|---:|---:|
| 1 | 0 | 0 | 0 |
| 2 | 0 | 1 | 0 |
| 3 | 1 | 0 | 1 |
| 4 | 1 | 1 | 1 |

### Correlation

$$\rho(x,y)=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}
{\sqrt{\sum_i(x_i-\bar x)^2\sum_i(y_i-\bar y)^2}}.$$

Use `effort` and `passed`. Both means are 0.5. Subtract the means,
sum the products and squared deviations, then substitute into the formula.

### Entropy and mutual information

$$H(X)=-\sum_x p(x)\log_2p(x),\qquad
I(X;Y)=H(X)+H(Y)-H(X,Y).$$

Use $\log_2(1/2)=-1$ and $\log_2(1/4)=-2$; skip zero-probability terms.
Compute $H(\text{passed})$. Each feature also has two zeros and two ones,
so it has the same entropy. Count the pairs for `effort` with `passed`,
then for `brno` with `passed`. Which feature removes uncertainty about passing?

*Write your calculations and answers here.*

## Task 3 — Implement the filter scores

The Pearson function is supplied. Implement **entropy** using
`np.unique(values, return_counts=True)`, then implement **mutual information**
from the formula in Task 2. Joint entropy is supplied.
Check both functions against your hand results and the libraries.

**Optional after Task 6:** reimplement Pearson correlation.

In [ ]:
hand = pd.DataFrame({"effort": [0, 0, 1, 1],
                     "brno": [0, 1, 0, 1],
                     "passed": [0, 0, 1, 1]})

def pearson(x, y):
    x = np.asarray(x) - np.mean(x)
    y = np.asarray(y) - np.mean(y)
    return np.sum(x * y) / np.sqrt(np.sum(x**2) * np.sum(y**2))

def entropy(values):
    # TODO: your code here
    ...

print("correlation:", pearson(hand["effort"], hand["passed"]))
print("entropy:", entropy(hand["passed"]))

Joint entropy is supplied. Complete mutual information using the entropy function you wrote. Empty bins are skipped before taking logs.

In [ ]:
def joint_entropy(a, b):
    counts = pd.crosstab(a, b).to_numpy()
    p = counts[counts > 0] / counts.sum()
    return -np.sum(p * np.log2(p))

def mutual_information(a, b):
    # TODO: your code here
    ...

from scipy.stats import pearsonr
from sklearn.metrics import mutual_info_score

print("Pearson, ours / SciPy:", pearson(hand["effort"], hand["passed"]),
      pearsonr(hand["effort"], hand["passed"]).statistic)
for name in ["effort", "brno"]:
    print(name, "MI in bits, ours / sklearn:",
          mutual_information(hand[name], hand["passed"]),
          mutual_info_score(hand[name], hand["passed"]) / np.log(2))

Scikit-learn uses natural logarithms; dividing by `np.log(2)` converts nats to bits. In this tiny table, effort determines the label and Brno provides no information about it.

**Optional after Task 6:** inspect the training-data correlation matrix. Which features look relevant or redundant? Skip this plot in the main session.

In [ ]:
#@title Correlation matrix (run this cell)
with_label = train.copy()
with_label["passed_exam"] = y_train
corr = with_label.corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr)), corr.columns)
fig.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

## Task 4 — The single best feature

A **filter** scores features without fitting a classifier. A **wrapper**
trains a classifier and measures its predictions.

1. Complete `knn_accuracy(columns)`: fit 3-NN on the selected training columns
   and return accuracy on the same validation columns.
2. Loop over all feature indices and append each single-feature accuracy to
   `single`. Pass `[j]` to keep a two-dimensional input for scikit-learn.

Which single feature works best? Reuse your scoring helper in Task 6.

In [ ]:
def knn_accuracy(columns):
    # TODO: your code here
    ...

single = []
# TODO: your code here
...

print(pd.Series(single, index=feature_names).sort_values(ascending=False))
print("all features:", knn_accuracy(list(range(len(feature_names)))))
majority = np.bincount(y_train).argmax()
baseline = np.mean(y_valid == majority)
print("majority-class baseline:", baseline)

Does using all columns improve on the best single column? Compare both
with the baseline. Each validation student changes accuracy by $1/14$;
small differences here are weak evidence.

## Task 5 — PCA (optional implementation)

**Optional after Task 6.** Implement the four-point example using the formulas
below. PCA is not required for the forward-selection task.

Selection keeps original columns. **PCA makes new features**, choosing
directions of large variance without using the labels.

Let $X\in\mathbb R^{n\times d}$ contain samples in rows and features in columns.
The steps follow the third lecture:

1. **Centre each feature:** $\bar x_j=\frac1n\sum_{i=1}^n X_{ij}$,
   $(X_c)_{ij}=X_{ij}-\bar x_j$.
2. **Covariance:** $S=\frac1n X_c^\top X_c\in\mathbb R^{d\times d}$.
3. **Principal directions:** $S\mathbf w_j=\lambda_j\mathbf w_j$,
   $\|\mathbf w_j\|_2=1$. Keep the directions with the largest eigenvalues.
4. **Projection:** $W_k=[\mathbf w_1,\ldots,\mathbf w_k]$,
   $Z=X_cW_k\in\mathbb R^{n\times k}$. For one component,
   $\mathbf z=X_c\mathbf w_1$.
5. **Variance retained:** $r_k=\dfrac{\lambda_1+\cdots+\lambda_k}
   {\lambda_1+\cdots+\lambda_d}$, with $\lambda_1\ge\cdots\ge\lambda_d$.

Implement these steps in NumPy. Store centred data in
`X_centered`, covariance in `S`, eigenvalues/eigenvectors in `eigenvalues, W`,
and the one-component projection in `z`.

`np.linalg.eigh(S)` returns **increasing** eigenvalues and matching eigenvectors
in the **columns** of `W`: use `W[:, -1]` for the largest eigenvalue.

In [ ]:
X_small = np.array([[2., 1.], [-2., -1.], [1., 2.], [-1., -2.]])
# TODO: your code here
...
print("covariance:", S)
print("eigenvalues:", eigenvalues)
print("first component:", z)
print("fraction of variance kept:", eigenvalues[-1] / eigenvalues.sum())

Check the result with scikit-learn. The eigenvalues should be 0.5 and 4.5,
so one component keeps 90% of the variance. A component may have its sign
reversed; the direction is still the same line.

In [ ]:
from sklearn.decomposition import PCA
pca_small = PCA(n_components=1).fit(X_small)
print("sklearn first component:", pca_small.transform(X_small).ravel())
print("sklearn variance ratio:", pca_small.explained_variance_ratio_)

**Optional continuation:** implement the PCA comparison on the student data
with **2 and 4 components**. Fit PCA on training data, transform training and
validation data, fit 3-NN, and report validation accuracy and retained variance.
Use `fit`, `transform` and `explained_variance_ratio_`.
This comparison and the plot below are not required for Task 6.

In [ ]:
for n_components in [2, 4]:
    # TODO: your code here
    ...

In [ ]:
#@title Show the first two components (run this cell)
Z = PCA(n_components=2).fit_transform(X_train)
for label, name in [(0, "failed"), (1, "passed")]:
    plt.scatter(Z[y_train == label, 0], Z[y_train == label, 1], label=name)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.legend()
plt.show()

Does keeping more variance necessarily improve classification?

## Task 6 — Forward selection


Start with no features. Try each remaining column, keep the best addition,
and repeat until validation accuracy stops improving.

**A quick board example.** These are invented validation scores for three
features A, B and C; no model fitting or arithmetic is needed on paper.
The no-feature baseline is 50%.

| subset | validation accuracy |
|---|---:|
| A | 70% |
| B | 60% |
| C | 50% |
| A, B | 80% |
| A, C | 70% |
| A, B, C | 80% |

Which feature comes first, which is added next, and when do we stop?

Now write the loop on the larger dataset. Call
`knn_accuracy(selected + [j])` for each remaining column `j`.

1. Collect one score per remaining feature.
2. Find the best score. `np.argmax(scores)` gives a **position in `remaining`**,
   not the original column number.
3. Stop if the best score does not improve on `best_score`.
4. Otherwise update the score, append the winning column to `selected`,
   remove it from `remaining`, and repeat.

Print the feature name and score after each accepted step.

In [ ]:
selected = []
remaining = list(range(len(feature_names)))
best_score = baseline

# TODO: your code here
...
print("selected:", [feature_names[j] for j in selected])

## Task 7 — Library comparison (teacher demonstration)

Scikit-learn can search in either direction. **Backward elimination** starts
with all columns and removes one at a time. Both searches are greedy, so
neither guarantees the best subset.

The supplied split tells scikit-learn to use exactly our training and
validation rows. This keeps the comparison with our loop simple; the test
set is not included. Here the library selects exactly two features.
Run the supplied code and compare the selected columns; no implementation is needed here.

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector

X_search = np.vstack([X_train, X_valid])
y_search = np.concatenate([y_train, y_valid])
split = [(np.arange(len(X_train)), np.arange(len(X_train), len(X_search)))]

for direction in ["forward", "backward"]:
    selector = SequentialFeatureSelector(
        KNeighborsClassifier(n_neighbors=3), n_features_to_select=2,
        direction=direction, scoring="accuracy", cv=split)
    selector.fit(X_search, y_search)
    columns = selector.get_support(indices=True)
    print(direction, [feature_names[j] for j in columns], knn_accuracy(columns))

Did the two searches keep the same columns? Our loop stops when no
addition improves the score; the library comparison keeps two columns even
if the second does not help.

**Final check.** Keep the subset from our forward loop as the final choice.
Evaluate it on the test set once, alongside the training-majority baseline.
Do not change the subset after seeing this score.

In [ ]:
if selected:
    model = KNeighborsClassifier(n_neighbors=3).fit(X_train[:, selected], y_train)
    prediction = model.predict(X_test[:, selected])
else:
    prediction = np.full(len(y_test), majority)
print("final test accuracy:", accuracy_score(y_test, prediction))
print("majority-class baseline:", np.mean(y_test == majority))
print("test samples:", len(y_test))

## Summary

- Filters score individual features; wrappers evaluate a model on subsets.
- Fit preprocessing on training data, choose features on validation data,
  and evaluate the final choice on test data.

**Two questions:** Why do we choose features using validation rather than test accuracy?
When should our forward-selection loop stop?

**Optional PCA question:** Why might PCA keep most of the variance but lose
useful information for classification?